# Attention Is All You Need 재구현 실습 - 추가 실습 코드: Multi-Head Attention 완전 구현

- Tutorial ID: `mod-attention-paper-lab`
- Tutorial: Attention Is All You Need 재구현 실습
- Section ID: `practice-multi-head-attention`
- Section: 추가 실습 코드: Multi-Head Attention 완전 구현


# Integrated Notebook: 04_Multi_Head_Attention

## Original Version
Source: 066_mod-attention-paper-lab_practice-multi-head-attention_Attention_Is_All_You_Need_#Uc7ac#Uad6c#Ud604_#Uc2e4#Uc2b5_-_#Ucd94#Uac00_#Uc2e4#Uc2b5_#Ucf54#Ub4dc-_Multi-Head_Attention_#Uc644#Uc804_#Uad6c#Ud604.ipynb

# Attention Is All You Need 재구현 실습 - 추가 실습 코드: Multi-Head Attention 완전 구현

- Tutorial ID: `mod-attention-paper-lab`
- Tutorial: Attention Is All You Need 재구현 실습
- Section: 추가 실습 코드: Multi-Head Attention 완전 구현

In [ ]:
# ============================================================
# 코드 읽는 법 — 추가 실습 코드: Multi-Head Attention 완전 구현
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

class MultiHeadAttention:
        def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 각 head의 dimension
        
        # Xavier 초기화
        scale = np.sqrt(2.0 / d_model)
        
        # 각 head의 Q, K, V 프로젝션 가중치
        self.W_q = np.random.randn(d_model, d_model) * scale
        self.W_k = np.random.randn(d_model, d_model) * scale
        self.W_v = np.random.randn(d_model, d_model) * scale
        
        # 출력 프로젝션
        self.W_o = np.random.randn(d_model, d_model) * scale
        
        print(f"Multi-Head Attention 초기화:")
        print(f"  - d_model: {d_model}")
        print(f"  - num_heads: {num_heads}")
        print(f"  - d_k (per head): {self.d_k}")
    
        def split_heads(self, x):
        """(seq_len, d_model) -> (num_heads, seq_len, d_k)"""
                seq_len = x.shape[0]
                x = x.reshape(seq_len, self.num_heads, self.d_k)
        return x.transpose(1, 0, 2)  # (num_heads, seq_len, d_k)
    
        def combine_heads(self, x):
        """(num_heads, seq_len, d_k) -> (seq_len, d_model)"""
                x = x.transpose(1, 0, 2)  # (seq_len, num_heads, d_k)
                seq_len = x.shape[0]
        return x.reshape(seq_len, self.d_model)
    
        def attention(self, Q, K, V, mask=None):
        """단일 head의 attention"""
        d_k = Q.shape[-1]
                scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(d_k)
        if mask is not None:
                        scores = np.where(mask == 0, -1e9, scores)
                weights = softmax(scores)
        return np.matmul(weights, V), weights
    
        def forward(self, X, mask=None):
        """
        Multi-Head Attention Forward Pass
        
        X: (seq_len, d_model)
        """
        # 1. Linear projections
                Q = np.matmul(X, self.W_q)
                K = np.matmul(X, self.W_k)
                V = np.matmul(X, self.W_v)
        
        # 2. Split into heads
        Q_heads = self.split_heads(Q)  # (num_heads, seq_len, d_k)
        K_heads = self.split_heads(K)
        V_heads = self.split_heads(V)
        
        # 3. Attention for each head
                attn_output, attn_weights = self.attention(Q_heads, K_heads, V_heads, mask)
        
        # 4. Combine heads
        combined = self.combine_heads(attn_output)  # (seq_len, d_model)
        
        # 5. Final linear projection
                output = np.matmul(combined, self.W_o)
        
        return output, attn_weights

# 테스트
np.random.seed(42)
seq_len = 4
d_model = 64
num_heads = 4

print("=" * 60)
print("Multi-Head Attention 테스트")
print("=" * 60)

# 입력 생성
X = np.random.randn(seq_len, d_model)
print(f"\\n입력 X shape: {X.shape}")

# MHA 생성 및 실행
mha = MultiHeadAttention(d_model, num_heads)

output, attn_weights = mha.forward(X)

print(f"\\n출력 shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"  - {num_heads}개의 head")
print(f"  - 각 head는 {seq_len}x{seq_len} attention matrix")

print("\\n각 Head의 Attention 패턴 (첫 행만 표시):")
for h in range(num_heads):
    print(f"  Head {h+1}: {attn_weights[h, 0].round(3)}")

print("\\n✅ Multi-Head Attention 완료!")
print("   각 head가 서로 다른 패턴을 학습합니다.")

## AI Developed Version 1
Source: 066_Multi-Head_Attention_#Uc644#Uc804_#Uad6c#Ud604.ipynb

# 📘 Multi-Head Attention 완전 구현

## "Attention Is All You Need" 재구현 실습 - Multi-Head Attention

---

### 🎯 학습 목표

1. Multi-Head Attention이 왜 Single-Head보다 좋은지 이해하기
2. Head를 나누고 합치는 과정을 단계별로 구현하기
3. 텐서의 shape 변화를 완벽히 추적하기
4. PyTorch `nn.MultiheadAttention`과 비교하기

### 📋 선수 지식

- 노트북 063~065의 Scaled Dot-Product Attention
- Causal Mask, Padding Mask

---

### 📖 배경: 왜 Multi-Head인가?

#### 비유: 여러 관점에서 바라보기

"나는 어제 **서울**에서 **친구**와 **맛있는** **라면**을 먹었다"

이 문장에서 "먹었다"는 여러 가지 관계를 동시에 가지고 있습니다:
- **Head 1** (누가?): "나는" → "먹었다" (주어-서술어)
- **Head 2** (무엇을?): "라면" → "먹었다" (목적어-서술어)
- **Head 3** (어떤?): "맛있는" → "라면" (수식어-피수식어)
- **Head 4** (어디서?): "서울" → "먹었다" (장소-행위)

Single-Head Attention은 이 중 하나의 관계만 잘 포착할 수 있지만,
**Multi-Head Attention은 여러 관계를 동시에 포착**할 수 있습니다!

### 📐 수식

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) \cdot W^O$$

$$\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
print(f"PyTorch 버전: {torch.__version__}")

## 1단계: Multi-Head의 직관적 이해

### Single-Head vs Multi-Head

**Single-Head** (d_model=8):
```
Q: (seq_len, 8) → 하나의 8차원 Attention
```

**Multi-Head** (d_model=8, num_heads=4):
```
Q: (seq_len, 8) → 4개의 2차원 Attention으로 분할
  Head 1: (seq_len, 2)  ← Q의 처음 2차원
  Head 2: (seq_len, 2)  ← Q의 다음 2차원
  Head 3: (seq_len, 2)  ← Q의 다음 2차원
  Head 4: (seq_len, 2)  ← Q의 마지막 2차원
```

핵심: **d_k = d_model / num_heads**

> 💡 총 연산량은 비슷하지만, 여러 "관점"에서 attention을 수행합니다!

In [ ]:
# ============================================================
# 하이퍼파라미터 설정

In [ ]:
d_model = 8        # 전체 임베딩 차원
num_heads = 4      # Head 수
d_k = d_model // num_heads  # 각 Head의 차원 = 8 / 4 = 2
d_v = d_k          # Value도 같은 차원
seq_len = 4        # 시퀀스 길이
batch_size = 1     # 배치 크기

print(f"d_model (전체 차원): {d_model}")
print(f"num_heads (Head 수): {num_heads}")
print(f"d_k (Head당 차원):   {d_k}")
print(f"d_model = num_heads × d_k = {num_heads} × {d_k} = {num_heads * d_k}")
print()
print("[중요] d_model이 num_heads로 나누어 떨어져야 합니다!")
assert d_model % num_heads == 0, "d_model은 num_heads의 배수여야 합니다"

## 2단계: 단계별 구현 - Head 분리하기

Multi-Head Attention의 가장 핵심적인 부분은
**텐서를 여러 Head로 나누고 다시 합치는 과정**입니다.

이 과정에서 `reshape`과 `transpose` 연산이 중요합니다.

In [ ]:
# ============================================================
# Step 2-1: 입력 생성 및 선형 변환

In [ ]:
# 입력
x = torch.randn(batch_size, seq_len, d_model)
print(f"입력 x shape: {x.shape}")
print(f"  → (batch={batch_size}, seq={seq_len}, d_model={d_model})")
print()

# 선형 변환 (전체 d_model 차원으로 변환)
# 주의: 각 Head별로 따로 만들지 않고, 하나의 큰 Linear로 처리!
W_Q = nn.Linear(d_model, d_model, bias=False)  # (8, 8)
W_K = nn.Linear(d_model, d_model, bias=False)
W_V = nn.Linear(d_model, d_model, bias=False)

Q = W_Q(x)  # (1, 4, 8)
K = W_K(x)  # (1, 4, 8)
V = W_V(x)  # (1, 4, 8)

print(f"Q shape (변환 후): {Q.shape}")
print(f"  → 아직 Head로 나뉘지 않은 상태")

In [ ]:
# ============================================================
# Step 2-2: Head 분리 (가장 중요한 단계!)

In [ ]:
# 핵심 아이디어:
# (batch, seq, d_model) → (batch, seq, num_heads, d_k) → (batch, num_heads, seq, d_k)
#
# 1. reshape(view): d_model을 num_heads × d_k로 분할
# 2. transpose: num_heads를 seq 앞으로 이동 (병렬 처리를 위해)

def split_heads(tensor, batch_size, num_heads, seq_len, d_k):
    """
    텐서를 여러 Head로 분리합니다.
    
    (batch, seq, d_model) → (batch, num_heads, seq, d_k)
    """
    # Step 1: 마지막 차원을 (num_heads, d_k)로 분할
    # (batch, seq, d_model) → (batch, seq, num_heads, d_k)
    tensor = tensor.view(batch_size, seq_len, num_heads, d_k)
    
    # Step 2: num_heads를 seq 앞으로 이동
    # (batch, seq, num_heads, d_k) → (batch, num_heads, seq, d_k)
    tensor = tensor.transpose(1, 2)
    
    return tensor


# Q를 Head별로 분리
print("Q shape 변환 과정:")
print(f"  원래:     {Q.shape}  (batch, seq, d_model)")

Q_split = Q.view(batch_size, seq_len, num_heads, d_k)
print(f"  view 후:  {Q_split.shape}  (batch, seq, num_heads, d_k)")

Q_heads = Q_split.transpose(1, 2)
print(f"  최종:     {Q_heads.shape}  (batch, num_heads, seq, d_k)")
print()

# K, V도 동일하게 분리
K_heads = split_heads(K, batch_size, num_heads, seq_len, d_k)
V_heads = split_heads(V, batch_size, num_heads, seq_len, d_k)

print(f"Q_heads: {Q_heads.shape}")
print(f"K_heads: {K_heads.shape}")
print(f"V_heads: {V_heads.shape}")
print()
print("[해석]")
print(f"  {num_heads}개의 Head가 각각 독립적으로 Attention을 수행합니다.")
print(f"  각 Head는 {d_k}차원의 Q, K, V를 사용합니다.")

In [ ]:
# ============================================================
# Step 2-3: 각 Head에서 독립적으로 Attention 수행

In [ ]:
# PyTorch의 브로드캐스팅 덕분에
# 4개 Head의 Attention이 한 번에 계산됩니다!

# 각 Head에서의 Attention Score
# (batch, heads, seq, d_k) @ (batch, heads, d_k, seq) = (batch, heads, seq, seq)
scores = Q_heads @ K_heads.transpose(-2, -1) / math.sqrt(d_k)

print(f"Scores shape: {scores.shape}")
print(f"  → (batch={batch_size}, heads={num_heads}, seq={seq_len}, seq={seq_len})")
print()

# Softmax
attn_weights = F.softmax(scores, dim=-1)

# 각 Head의 Attention weights 확인
for h in range(num_heads):
    print(f"Head {h+1} Attention Weights:")
    print(attn_weights[0, h].detach())
    print()

print("[관찰] 각 Head마다 다른 attention 패턴을 보입니다!")
print("  이것이 Multi-Head의 장점: 다양한 관계를 동시에 포착")

In [ ]:
# ============================================================
# Step 2-4: Value에 가중치 적용

In [ ]:
# (batch, heads, seq, seq) @ (batch, heads, seq, d_k) = (batch, heads, seq, d_k)
head_outputs = attn_weights @ V_heads

print(f"각 Head의 출력 shape: {head_outputs.shape}")
print(f"  → (batch={batch_size}, heads={num_heads}, seq={seq_len}, d_k={d_k})")

## 3단계: Head 합치기 (Concatenation)

각 Head의 출력을 다시 합쳐서 원래 d_model 차원으로 되돌립니다.

```
(batch, num_heads, seq, d_k) → (batch, seq, num_heads, d_k) → (batch, seq, d_model)
```

Split의 역순입니다!

In [ ]:
# ============================================================
# Head 합치기

In [ ]:
def merge_heads(tensor, batch_size, num_heads, seq_len, d_k):
    """
    분리된 Head들을 다시 합칩니다.
    split_heads의 역순입니다.
    
    (batch, num_heads, seq, d_k) → (batch, seq, d_model)
    """
    # Step 1: num_heads를 seq 뒤로 이동
    # (batch, num_heads, seq, d_k) → (batch, seq, num_heads, d_k)
    tensor = tensor.transpose(1, 2)
    
    # Step 2: num_heads와 d_k를 합쳐서 d_model로
    # (batch, seq, num_heads, d_k) → (batch, seq, d_model)
    # contiguous(): 메모리 레이아웃을 연속적으로 만듦 (transpose 후 필요)
    tensor = tensor.contiguous().view(batch_size, seq_len, num_heads * d_k)
    
    return tensor


# Head 합치기 과정
print("Head 합치기 과정:")
print(f"  Head 출력:  {head_outputs.shape}  (batch, heads, seq, d_k)")

step1 = head_outputs.transpose(1, 2)
print(f"  transpose: {step1.shape}  (batch, seq, heads, d_k)")

concat_output = step1.contiguous().view(batch_size, seq_len, d_model)
print(f"  합치기:    {concat_output.shape}  (batch, seq, d_model)")
print()
print(f"[확인] d_model = num_heads × d_k = {num_heads} × {d_k} = {d_model} ✅")

In [ ]:
# ============================================================
# 최종 출력 선형 변환 (W_O)

In [ ]:
# 합쳐진 Head 출력에 마지막 선형 변환을 적용합니다.
# 이것이 논문의 W^O에 해당합니다.
W_O = nn.Linear(d_model, d_model, bias=False)

final_output = W_O(concat_output)

print(f"최종 출력 shape: {final_output.shape}")
print(f"  → 입력과 동일한 shape! (batch, seq, d_model)")

## 4단계: 전체를 하나의 클래스로 구현

지금까지의 모든 단계를 깔끔한 nn.Module 클래스로 정리합니다.

In [ ]:
# ============================================================
# Multi-Head Attention 클래스 (완전 구현)

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention 레이어.
    
    "Attention Is All You Need" 논문의 Multi-Head Attention을
    그대로 구현한 것입니다.
    
    Args:
        d_model: 입력/출력 차원
        num_heads: Head 수 (d_model의 약수여야 함)
        dropout: Attention weight에 적용할 dropout 비율
    """
    
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        
        # d_model이 num_heads로 나누어 떨어지는지 확인
        assert d_model % num_heads == 0, \
            f"d_model({d_model})은 num_heads({num_heads})의 배수여야 합니다"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Head당 차원
        
        # 선형 변환 레이어들
        self.W_Q = nn.Linear(d_model, d_model)  # Q 변환
        self.W_K = nn.Linear(d_model, d_model)  # K 변환
        self.W_V = nn.Linear(d_model, d_model)  # V 변환
        self.W_O = nn.Linear(d_model, d_model)  # 출력 변환
        
        # Dropout (과적합 방지)
        self.dropout = nn.Dropout(dropout)
        
        # 스케일 팩터
        self.scale = math.sqrt(self.d_k)
    
    def split_heads(self, x):
        """(batch, seq, d_model) → (batch, num_heads, seq, d_k)"""
        batch_size, seq_len, _ = x.shape
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        return x.transpose(1, 2)
    
    def merge_heads(self, x):
        """(batch, num_heads, seq, d_k) → (batch, seq, d_model)"""
        batch_size, _, seq_len, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(batch_size, seq_len, self.d_model)
    
    def forward(self, query, key, value, mask=None):
        """
        Args:
            query: (batch_size, seq_len_q, d_model)
            key:   (batch_size, seq_len_k, d_model)
            value: (batch_size, seq_len_v, d_model)
            mask:  (batch_size, 1, 1, seq_len_k) 또는
                   (batch_size, 1, seq_len_q, seq_len_k)
        
        Returns:
            output: (batch_size, seq_len_q, d_model)
            attn_weights: (batch_size, num_heads, seq_len_q, seq_len_k)
        
        참고: Self-Attention에서는 query=key=value=x 입니다.
              Cross-Attention에서는 query≠key=value 입니다.
        """
        # 1. 선형 변환
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)
        
        # 2. Head 분리
        Q = self.split_heads(Q)  # (batch, heads, seq_q, d_k)
        K = self.split_heads(K)  # (batch, heads, seq_k, d_k)
        V = self.split_heads(V)  # (batch, heads, seq_v, d_k)
        
        # 3. Scaled Dot-Product Attention
        scores = (Q @ K.transpose(-2, -1)) / self.scale
        
        # 4. Mask 적용 (선택적)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # 5. Softmax + Dropout
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # 6. Value에 가중치 적용
        context = attn_weights @ V  # (batch, heads, seq_q, d_k)
        
        # 7. Head 합치기
        context = self.merge_heads(context)  # (batch, seq_q, d_model)
        
        # 8. 출력 선형 변환
        output = self.W_O(context)
        
        return output, attn_weights


# 테스트
print("=" * 60)
print("Multi-Head Attention 테스트")
print("=" * 60)

d_model_test = 64
num_heads_test = 8
mha = MultiHeadAttention(d_model_test, num_heads_test)

x_test = torch.randn(2, 6, d_model_test)  # 배치 2, 시퀀스 6

# Self-Attention (query=key=value=x)
output_mha, weights_mha = mha(x_test, x_test, x_test)

print(f"입력 shape:  {x_test.shape}")
print(f"출력 shape:  {output_mha.shape}")
print(f"가중치 shape: {weights_mha.shape}")
print(f"  → (batch=2, heads={num_heads_test}, seq=6, seq=6)")
print()

# 모델 파라미터 수
total_params = sum(p.numel() for p in mha.parameters())
print(f"총 파라미터 수: {total_params:,}")
print(f"  W_Q: {d_model_test}×{d_model_test} + {d_model_test} (bias) = {d_model_test*d_model_test + d_model_test}")
print(f"  × 4 (W_Q, W_K, W_V, W_O) = {(d_model_test*d_model_test + d_model_test) * 4}")

## 5단계: 각 Head의 Attention 패턴 시각화

각 Head가 서로 다른 attention 패턴을 학습하는 것을 확인합니다.

In [ ]:
# ============================================================
# 각 Head의 Attention 패턴 시각화

In [ ]:
# 작은 모델로 시각화
d_model_viz = 16
num_heads_viz = 4
seq_len_viz = 5

mha_viz = MultiHeadAttention(d_model_viz, num_heads_viz)
x_viz = torch.randn(1, seq_len_viz, d_model_viz)
_, weights_viz = mha_viz(x_viz, x_viz, x_viz)

tokens_viz = ["나는", "오늘", "학교에", "갔다", "."]

fig, axes = plt.subplots(1, num_heads_viz, figsize=(4 * num_heads_viz, 4))

for h in range(num_heads_viz):
    ax = axes[h]
    w = weights_viz[0, h].detach().numpy()
    im = ax.imshow(w, cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(seq_len_viz))
    ax.set_yticks(range(seq_len_viz))
    ax.set_xticklabels(tokens_viz, rotation=45, fontsize=9)
    ax.set_yticklabels(tokens_viz, fontsize=9)
    ax.set_title(f"Head {h+1}", fontsize=13)
    
    for i in range(seq_len_viz):
        for j in range(seq_len_viz):
            color = "white" if w[i, j] > 0.5 else "black"
            ax.text(j, i, f"{w[i,j]:.2f}", ha="center", va="center",
                    fontsize=8, color=color)

plt.suptitle("Multi-Head Attention - 각 Head의 Attention 패턴", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("[관찰]")
print("  각 Head가 서로 다른 attention 패턴을 보입니다.")
print("  학습이 진행되면, 각 Head는 서로 다른 유형의 관계를 포착하게 됩니다.")
print("  예: 구문 관계, 의미 관계, 거리 관계 등")

## 6단계: Cross-Attention 이해하기

지금까지는 **Self-Attention** (query=key=value)만 다뤘습니다.
Transformer에는 **Cross-Attention**도 있습니다.

### Self-Attention vs Cross-Attention

| | Self-Attention | Cross-Attention |
|--|---------------|----------------|
| Q, K, V 출처 | 모두 같은 입력 | Q≠K=V (다른 입력) |
| 용도 | 자신의 문맥 파악 | 다른 시퀀스와 관계 파악 |
| 예시 | 문장 내 단어 관계 | 번역 시 원문↔번역문 관계 |
| 위치 | Encoder, Decoder | Decoder |

번역 예시:
- Encoder 입력: "I love cats" (영어)
- Decoder 입력: "나는 고양이를 좋아한다" (한국어)
- Cross-Attention: Decoder가 Encoder의 출력을 참조

In [ ]:
# ============================================================
# Cross-Attention 예시

In [ ]:
# Encoder 출력 (영어 문장)
encoder_output = torch.randn(1, 3, d_model_viz)  # "I love cats" (3 토큰)

# Decoder 입력 (한국어 문장)
decoder_input = torch.randn(1, 4, d_model_viz)  # "나는 고양이를 좋아 한다" (4 토큰)

# Cross-Attention: Q=decoder, K=V=encoder
cross_attn = MultiHeadAttention(d_model_viz, num_heads_viz)
cross_output, cross_weights = cross_attn(
    query=decoder_input,    # Q: 번역문 (질문하는 쪽)
    key=encoder_output,     # K: 원문 (답변의 색인)
    value=encoder_output    # V: 원문 (답변의 내용)
)

print(f"Decoder 입력 shape:     {decoder_input.shape}  (4 토큰)")
print(f"Encoder 출력 shape:     {encoder_output.shape}  (3 토큰)")
print(f"Cross-Attention 출력:   {cross_output.shape}  (4 토큰, encoder 정보 반영)")
print(f"Cross-Attention 가중치: {cross_weights.shape}")
print(f"  → (batch=1, heads={num_heads_viz}, decoder_seq=4, encoder_seq=3)")
print()
print("[핵심]")
print("  Cross-Attention의 가중치 shape이 (4, 3)인 것에 주목!")
print("  Decoder의 4개 토큰이 Encoder의 3개 토큰을 참조합니다.")
print("  → 번역 시 각 한국어 단어가 어떤 영어 단어에 대응되는지 학습!")

## 🔑 핵심 정리

### Multi-Head Attention 전체 흐름

```
입력 x: (batch, seq, d_model)
  │
  ├─ W_Q ─→ Q: (batch, seq, d_model)
  ├─ W_K ─→ K: (batch, seq, d_model)
  └─ W_V ─→ V: (batch, seq, d_model)
  │
  ▼ split_heads
  Q, K, V: (batch, num_heads, seq, d_k)
  │
  ▼ Scaled Dot-Product Attention (각 Head 독립)
  head_outputs: (batch, num_heads, seq, d_k)
  │
  ▼ merge_heads (Concat)
  concat: (batch, seq, d_model)
  │
  ▼ W_O
  출력: (batch, seq, d_model)
```

### Shape 변화 요약 (d_model=512, num_heads=8, d_k=64)

| 단계 | Shape |
|------|-------|
| 입력 | (batch, seq, 512) |
| Q, K, V (변환 후) | (batch, seq, 512) |
| Head 분리 후 | (batch, 8, seq, 64) |
| Attention scores | (batch, 8, seq, seq) |
| Head 합치기 후 | (batch, seq, 512) |
| 최종 출력 | (batch, seq, 512) |

### 다음 단계

- **노트북 067**: BERT의 Masked Language Modeling
- **노트북 068**: MLM 마스킹 구현

## AI Developed Version 2
Source: 04_Multi_Head_Attention.ipynb

# 📚 Notebook 4: Multi-Head Attention 완전 구현

## "Attention Is All You Need" 재구현 - Multi-Head Attention

---

### 🎯 학습 목표
Transformer의 핵심 구조인 **Multi-Head Attention**을 완전히 구현합니다.

### 💡 왜 Multi-Head인가?

하나의 Attention(Single Head)은 하나의 관점에서만 관계를 파악합니다.
하지만 언어에는 여러 종류의 관계가 있습니다!

```
예: "그 고양이가 매트 위에서 잠을 잤다"

Head 1: 문법적 관계 파악 → "고양이가" ↔ "잤다" (주어-동사)
Head 2: 위치 관계 파악   → "매트" ↔ "위에서" (장소)
Head 3: 수식 관계 파악   → "그" ↔ "고양이가" (지시어-명사)
```

여러 Head가 각각 다른 관점에서 Attention을 수행하면 더 풍부한 정보를 얻을 수 있습니다!

### 📋 목차
1. Single Head vs Multi-Head 비교
2. Multi-Head Attention 수식 이해
3. Step-by-step 구현
4. PyTorch nn.Module로 완성
5. 각 Head의 Attention 패턴 시각화
6. PyTorch 내장 MultiheadAttention과 비교

---

In [ ]:
# ============================================================
# 환경 설정

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
print("✅ 라이브러리 로딩 완료!")

## 1. Single Head vs Multi-Head 비교

### 수식 비교

**Single-Head Attention:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Multi-Head Attention:**
$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$
$$\text{where } \text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

### 차원 분배
```
d_model = 512 (모델의 전체 차원)
num_heads = 8 (Head 수)
d_k = d_v = d_model / num_heads = 64 (각 Head의 차원)
```

전체 차원을 Head 수로 나누어 각 Head에 분배합니다.
→ 계산량은 Single Head와 거의 같으면서 더 다양한 패턴을 학습!

In [ ]:
# ============================================================
# Multi-Head Attention의 차원 분배 이해

In [ ]:
# Transformer 논문의 기본 설정
d_model = 512    # 모델의 기본 차원
num_heads = 8    # Head 수
d_k = d_model // num_heads  # 각 Head의 Query/Key 차원
d_v = d_model // num_heads  # 각 Head의 Value 차원

print("📐 Multi-Head Attention 차원 설계")
print("=" * 50)
print(f"d_model (전체 차원):     {d_model}")
print(f"num_heads (Head 수):     {num_heads}")
print(f"d_k (각 Head의 Q/K 차원): {d_k}  (= {d_model} ÷ {num_heads})")
print(f"d_v (각 Head의 V 차원):   {d_v}  (= {d_model} ÷ {num_heads})")
print(f"\n총 파라미터 수 비교:")
print(f"  Single Head:  {d_model} × {d_model} × 3 = {d_model * d_model * 3:,}")
print(f"  Multi Head:   {d_model} × {d_k} × {num_heads} × 3 = {d_model * d_k * num_heads * 3:,}")
print(f"  → 같은 파라미터 수!")

## 2. Step-by-Step 구현

작은 예시(d_model=16, num_heads=4)로 각 단계를 차근차근 구현해봅시다.

### 전체 흐름:
```
입력 x (batch, seq_len, d_model)
  ↓
Linear 변환 → Q, K, V  (batch, seq_len, d_model)
  ↓
Head로 분리 (reshape)   (batch, num_heads, seq_len, d_k)
  ↓
각 Head에서 Attention   (batch, num_heads, seq_len, d_v)
  ↓
Head 결합 (concat)      (batch, seq_len, d_model)
  ↓
출력 Linear 변환         (batch, seq_len, d_model)
```

In [ ]:
# ============================================================
# Step 1: 입력 데이터 및 Linear 변환

In [ ]:
# 작은 예시 설정
batch_size = 2
seq_len = 4
d_model = 16
num_heads = 4
d_k = d_model // num_heads  # = 4
d_v = d_model // num_heads  # = 4

print(f"설정: batch={batch_size}, seq_len={seq_len}, d_model={d_model}, heads={num_heads}, d_k={d_k}")

# 입력 데이터
x = torch.randn(batch_size, seq_len, d_model)
print(f"\nStep 1: 입력 x shape = {x.shape}")
print(f"  → (배치={batch_size}) × (시퀀스={seq_len}) × (차원={d_model})")

# Q, K, V를 위한 Linear 변환
# 각각 d_model → d_model 크기의 선형 변환
W_Q = nn.Linear(d_model, d_model, bias=False)
W_K = nn.Linear(d_model, d_model, bias=False)
W_V = nn.Linear(d_model, d_model, bias=False)

# Linear 변환 적용
Q = W_Q(x)  # (batch, seq_len, d_model)
K = W_K(x)
V = W_V(x)

print(f"\nLinear 변환 후:")
print(f"  Q shape = {Q.shape}  (아직 Head로 분리 전)")
print(f"  K shape = {K.shape}")
print(f"  V shape = {V.shape}")

In [ ]:
# ============================================================
# Step 2: Head로 분리 (Reshape + Transpose)

In [ ]:
# 핵심 아이디어:
# d_model = num_heads × d_k 이므로,
# 마지막 차원을 (num_heads, d_k)로 나눌 수 있습니다.

# (batch, seq_len, d_model) → (batch, seq_len, num_heads, d_k)
Q_reshaped = Q.view(batch_size, seq_len, num_heads, d_k)
print(f"Step 2a: reshape 후 Q shape = {Q_reshaped.shape}")
print(f"  (배치={batch_size}, 시퀀스={seq_len}, 헤드={num_heads}, 헤드차원={d_k})")

# (batch, seq_len, num_heads, d_k) → (batch, num_heads, seq_len, d_k)
# → Head 차원을 앞으로 이동시켜서 각 Head가 독립적으로 Attention 수행
Q_heads = Q_reshaped.transpose(1, 2)
print(f"\nStep 2b: transpose 후 Q shape = {Q_heads.shape}")
print(f"  (배치={batch_size}, 헤드={num_heads}, 시퀀스={seq_len}, 헤드차원={d_k})")

# K, V도 동일하게 처리
K_heads = K.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
V_heads = V.view(batch_size, seq_len, num_heads, d_v).transpose(1, 2)

print(f"\nK_heads shape = {K_heads.shape}")
print(f"V_heads shape = {V_heads.shape}")

print(f"\n💡 이제 각 Head는 독립적인 (seq_len, d_k) 크기의 Q, K, V를 가집니다!")

In [ ]:
# ============================================================
# Step 3: 각 Head에서 Scaled Dot-Product Attention 수행

In [ ]:
# 배치 + 멀티헤드를 한 번에 처리!
# PyTorch의 matmul은 마지막 2개 차원에서 행렬 곱을 수행하고,
# 나머지 차원(batch, heads)은 자동으로 배치 처리합니다.

# QK^T / √d_k
scores = torch.matmul(Q_heads, K_heads.transpose(-2, -1)) / math.sqrt(d_k)
print(f"Step 3a: 유사도 점수 shape = {scores.shape}")
print(f"  (배치={batch_size}, 헤드={num_heads}, 시퀀스={seq_len}, 시퀀스={seq_len})")
print(f"  → 각 Head마다 {seq_len}×{seq_len} 유사도 행렬!")

# Softmax
attention_weights = F.softmax(scores, dim=-1)
print(f"\nStep 3b: Attention 가중치 shape = {attention_weights.shape}")

# 가중합
head_outputs = torch.matmul(attention_weights, V_heads)
print(f"\nStep 3c: Head 출력 shape = {head_outputs.shape}")
print(f"  → 각 Head의 출력: (시퀀스={seq_len}, 헤드차원={d_v})")

In [ ]:
# ============================================================
# Step 4: Head 결합 (Concatenation)

In [ ]:
# (batch, num_heads, seq_len, d_v) → (batch, seq_len, num_heads, d_v)
head_outputs_transposed = head_outputs.transpose(1, 2)
print(f"Step 4a: transpose 후 shape = {head_outputs_transposed.shape}")

# (batch, seq_len, num_heads, d_v) → (batch, seq_len, num_heads * d_v)
# = (batch, seq_len, d_model)
# contiguous(): 메모리 레이아웃을 연속적으로 재배치 (view 사용을 위해 필요)
concatenated = head_outputs_transposed.contiguous().view(batch_size, seq_len, d_model)
print(f"Step 4b: concatenation 후 shape = {concatenated.shape}")
print(f"  → ({num_heads}개 Head × {d_v}차원) = {d_model}차원으로 합쳐짐!")

In [ ]:
# ============================================================
# Step 5: 출력 Linear 변환 (W^O)

In [ ]:
# 결합된 Head 출력에 최종 Linear 변환 적용
# 이 변환은 여러 Head의 정보를 통합하는 역할
W_O = nn.Linear(d_model, d_model, bias=False)
output = W_O(concatenated)

print(f"Step 5: 최종 출력 shape = {output.shape}")
print(f"  → 입력과 동일한 shape! ({batch_size}, {seq_len}, {d_model})")
print(f"\n✅ Multi-Head Attention 전체 과정 완료!")

## 3. 전체를 하나의 nn.Module로 구현

위의 모든 단계를 깔끔하게 하나의 클래스로 합쳐봅시다.

In [ ]:
# ============================================================
# Multi-Head Attention 완전 구현

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention 구현
    
    MultiHead(Q, K, V) = Concat(head_1, ..., head_h) W^O
    where head_i = Attention(Q W_i^Q, K W_i^K, V W_i^V)
    """
    
    def __init__(self, d_model, num_heads):
        """
        Args:
            d_model (int): 모델의 전체 차원
            num_heads (int): Head의 수
        """
        super().__init__()
        
        # d_model이 num_heads로 나누어 떨어져야 합니다
        assert d_model % num_heads == 0, \
            f"d_model({d_model})은 num_heads({num_heads})로 나누어 떨어져야 합니다!"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 각 Head의 차원
        
        # Q, K, V를 위한 Linear 변환
        # 실제로는 하나의 큰 Linear로 한 번에 처리하는 것이 효율적
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        
        # 출력 Linear 변환 (W^O)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
    
    def split_heads(self, x):
        """
        텐서를 여러 Head로 분리합니다.
        
        (batch, seq_len, d_model) → (batch, num_heads, seq_len, d_k)
        """
        batch_size, seq_len, _ = x.size()
        # d_model → (num_heads, d_k)로 reshape
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        # (batch, seq_len, heads, d_k) → (batch, heads, seq_len, d_k)
        return x.transpose(1, 2)
    
    def combine_heads(self, x):
        """
        여러 Head의 출력을 결합합니다.
        
        (batch, num_heads, seq_len, d_k) → (batch, seq_len, d_model)
        """
        batch_size, _, seq_len, _ = x.size()
        # (batch, heads, seq_len, d_k) → (batch, seq_len, heads, d_k)
        x = x.transpose(1, 2)
        # (batch, seq_len, heads, d_k) → (batch, seq_len, d_model)
        return x.contiguous().view(batch_size, seq_len, self.d_model)
    
    def forward(self, query, key, value, mask=None):
        """
        Args:
            query (Tensor): (batch, seq_len_q, d_model)
            key (Tensor): (batch, seq_len_k, d_model)
            value (Tensor): (batch, seq_len_v, d_model)
            mask (Tensor, optional): 마스크 텐서
        
        Returns:
            output: (batch, seq_len_q, d_model)
            attention_weights: (batch, num_heads, seq_len_q, seq_len_k)
        """
        # Step 1: Linear 변환
        Q = self.W_Q(query)  # (batch, seq_len, d_model)
        K = self.W_K(key)
        V = self.W_V(value)
        
        # Step 2: Head 분리
        Q = self.split_heads(Q)  # (batch, heads, seq_len, d_k)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # Step 3: Scaled Dot-Product Attention (각 Head에서)
        d_k = self.d_k
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        
        if mask is not None:
            # mask shape: (batch, 1, 1, seq_len) 또는 (batch, 1, seq_len, seq_len)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attention_weights = F.softmax(scores, dim=-1)
        head_outputs = torch.matmul(attention_weights, V)
        
        # Step 4: Head 결합
        combined = self.combine_heads(head_outputs)  # (batch, seq_len, d_model)
        
        # Step 5: 출력 Linear 변환
        output = self.W_O(combined)
        
        return output, attention_weights

print("✅ MultiHeadAttention 클래스 정의 완료!")

In [ ]:
# ============================================================
# 테스트

In [ ]:
d_model = 32
num_heads = 4
batch_size = 2
seq_len = 6

# 모듈 생성
mha = MultiHeadAttention(d_model, num_heads)

# 입력 데이터 (Self-Attention: query = key = value)
x = torch.randn(batch_size, seq_len, d_model)

# Forward pass
output, attn_weights = mha(x, x, x)

print(f"입력 shape:      {x.shape}")
print(f"출력 shape:      {output.shape}")
print(f"가중치 shape:    {attn_weights.shape}")
print(f"  → (배치={batch_size}, 헤드={num_heads}, 시퀀스={seq_len}, 시퀀스={seq_len})")

# 파라미터 수 확인
total_params = sum(p.numel() for p in mha.parameters())
print(f"\n총 파라미터 수: {total_params:,}")
print(f"  W_Q: {d_model}×{d_model} = {d_model*d_model}")
print(f"  W_K: {d_model}×{d_model} = {d_model*d_model}")
print(f"  W_V: {d_model}×{d_model} = {d_model*d_model}")
print(f"  W_O: {d_model}×{d_model} = {d_model*d_model}")
print(f"  합계: {d_model*d_model*4}")

## 4. 각 Head의 Attention 패턴 시각화 🎨

Multi-Head Attention의 핵심 장점은 각 Head가 **서로 다른 패턴**을 학습한다는 것입니다.

In [ ]:
# ============================================================
# 각 Head별 Attention 가중치 시각화

In [ ]:
tokens = ['나는', '그', '작은', '고양이를', '매우', '좋아한다']

# 첫 번째 배치의 Attention 가중치 시각화
fig, axes = plt.subplots(1, num_heads, figsize=(20, 4))

for head_idx in range(num_heads):
    w = attn_weights[0, head_idx].detach().numpy()  # 첫 번째 배치, head_idx번 Head
    
    axes[head_idx].imshow(w, cmap='Blues', vmin=0, vmax=w.max())
    axes[head_idx].set_title(f'Head {head_idx + 1}', fontsize=14, fontweight='bold')
    axes[head_idx].set_xticks(range(seq_len))
    axes[head_idx].set_yticks(range(seq_len))
    axes[head_idx].set_xticklabels(tokens, fontsize=8, rotation=45)
    axes[head_idx].set_yticklabels(tokens, fontsize=8)
    
    if head_idx == 0:
        axes[head_idx].set_ylabel('Query', fontsize=12)
    axes[head_idx].set_xlabel('Key', fontsize=12)

plt.suptitle('각 Head의 Attention 패턴 (초기 - 학습 전)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 학습 전이라 패턴이 비슷하지만, 학습 후에는 각 Head가 다른 패턴을 보입니다.")
print("   예: Head 1 → 문법 관계, Head 2 → 인접 단어, Head 3 → 장거리 의존성 등")

In [ ]:
# ============================================================
# Causal Mask와 함께 Multi-Head Attention 사용

In [ ]:
# Causal Mask 생성 (Decoder용)
causal_mask = torch.tril(torch.ones(seq_len, seq_len))
# Multi-Head Attention에 맞는 shape으로 변환
# (seq_len, seq_len) → (1, 1, seq_len, seq_len)
causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

# Causal Attention 수행
output_causal, weights_causal = mha(x, x, x, mask=causal_mask)

# 시각화
fig, axes = plt.subplots(1, num_heads, figsize=(20, 4))
for head_idx in range(num_heads):
    w = weights_causal[0, head_idx].detach().numpy()
    axes[head_idx].imshow(w, cmap='Oranges', vmin=0, vmax=w.max())
    axes[head_idx].set_title(f'Head {head_idx + 1} (Causal)', fontsize=14, fontweight='bold')
    axes[head_idx].set_xticks(range(seq_len))
    axes[head_idx].set_yticks(range(seq_len))
    axes[head_idx].set_xticklabels(tokens, fontsize=8, rotation=45)
    axes[head_idx].set_yticklabels(tokens, fontsize=8)

plt.suptitle('Causal Multi-Head Attention 패턴', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 모든 Head에서 오른쪽 상단(미래)이 0인 것을 확인!")

## 5. PyTorch 내장 MultiheadAttention과 비교

PyTorch에는 이미 `nn.MultiheadAttention`이 구현되어 있습니다.
우리의 구현과 비교해봅시다.

In [ ]:
# ============================================================
# PyTorch 내장 vs 우리의 구현 비교

In [ ]:
# PyTorch 내장 MultiheadAttention
# 주의: PyTorch의 MHA는 입력 shape이 (seq_len, batch, d_model)
# batch_first=True로 설정하면 (batch, seq_len, d_model)
pytorch_mha = nn.MultiheadAttention(
    embed_dim=d_model,
    num_heads=num_heads,
    batch_first=True,   # 우리의 구현과 같은 shape 사용
    bias=False
)

# 같은 입력으로 테스트
output_pytorch, weights_pytorch = pytorch_mha(x, x, x)

print("비교 결과:")
print(f"  우리의 구현 출력 shape: {output.shape}")
print(f"  PyTorch 내장 출력 shape: {output_pytorch.shape}")
print(f"\n  Shape이 동일합니다! ✅")
print(f"\n  (값은 가중치 초기화가 다르므로 다를 수 있습니다)")

# 파라미터 수 비교
our_params = sum(p.numel() for p in mha.parameters())
pytorch_params = sum(p.numel() for p in pytorch_mha.parameters())
print(f"\n  우리의 파라미터 수: {our_params:,}")
print(f"  PyTorch 파라미터 수: {pytorch_params:,}")

## 📝 핵심 정리

### Multi-Head Attention 구현 핵심:

| 단계 | 연산 | Shape 변화 |
|------|------|------------|
| 1 | Linear 변환 | (B, S, D) → (B, S, D) |
| 2 | Head 분리 | (B, S, D) → (B, H, S, D/H) |
| 3 | Attention | (B, H, S, D/H) → (B, H, S, D/H) |
| 4 | Head 결합 | (B, H, S, D/H) → (B, S, D) |
| 5 | 출력 변환 | (B, S, D) → (B, S, D) |

여기서 B=배치, S=시퀀스, D=d_model, H=num_heads

### 다음 노트북 예고 🔜
Notebook 5에서는 **BERT의 Masked Language Modeling (MLM)**을 구현합니다.
→ 양방향 사전학습의 핵심 기법!

## AI Developed Version 3
Source: 066_Multi_Head_Attention_Complete.ipynb

# 📘 실습 066: Multi-Head Attention 완전 구현

## 🎯 왜 Multi-Head가 필요할까?

Single-Head Attention의 한계:
```
문장: "The bank can guarantee deposits will eventually cover future tuition costs."

Single Head: 'bank'를 볼 때 한 가지 관계만 학습
  → '금융기관'에 집중? 아니면 '강둑'에 집중? 하나만 가능!
```

Multi-Head Attention의 장점:
```
Multi-Head (8개 헤드):
  Head 1: 문법적 관계 학습 (주어-동사 관계)
  Head 2: 의미적 관계 학습 (유사어, 반의어)
  Head 3: 장거리 의존성 학습
  Head 4~8: 다른 패턴들...
  → 여러 관계를 동시에 포착!
```

## 📐 Multi-Head Attention 공식

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$
$$\text{where } \text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

### 학습 목표
1. Multi-Head Attention 구조 이해
2. 여러 헤드를 효율적으로 병렬 처리하는 방법
3. 완전한 Multi-Head Attention 클래스 구현
4. Transformer 인코더 블록 구현

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ 라이브러리 로드 완료 | 디바이스: {device}")

## 🎯 Step 1: Multi-Head의 핵심 아이디어

d_model=512, num_heads=8 일 때:
- 각 헤드: d_k = d_model / num_heads = 512/8 = **64**
- 8개 헤드가 각각 64차원에서 Attention 계산
- 결과를 합쳐(Concat) 다시 512차원으로!

In [ ]:
# ============================================================
# 🎯 Step 1: 단순한 방법 vs 효율적인 방법

In [ ]:
#
# 방법 1: 단순 방법 (이해하기 쉽지만 비효율적)
#   → 헤드마다 별도의 W_Q, W_K, W_V 생성
#   → 루프로 헤드별 계산
#
# 방법 2: 효율적인 방법 (실제 구현에서 사용)
#   → 하나의 큰 W_Q, W_K, W_V로 모든 헤드를 한번에 계산
#   → 행렬 reshape으로 헤드 분리
#   → GPU 병렬 연산 최대 활용

print("=" * 60)
print("방법 1: 이해하기 쉬운 구현 (head별 별도 계산)")
print("=" * 60)

d_model = 512
num_heads = 8
d_k = d_model // num_heads  # 64
batch_size = 2
seq_len = 5

print(f"d_model = {d_model}")
print(f"num_heads = {num_heads}")
print(f"d_k per head = d_model // num_heads = {d_model} // {num_heads} = {d_k}")
print()

# 각 헤드별 W_Q, W_K, W_V (총 8*3=24개 행렬)
heads_WQ = [nn.Linear(d_model, d_k, bias=False) for _ in range(num_heads)]
heads_WK = [nn.Linear(d_model, d_k, bias=False) for _ in range(num_heads)]
heads_WV = [nn.Linear(d_model, d_k, bias=False) for _ in range(num_heads)]

x = torch.randn(batch_size, seq_len, d_model)

# 각 헤드 별도 계산
head_outputs = []
for h in range(num_heads):
    Q_h = heads_WQ[h](x)  # (batch, seq, d_k)
    K_h = heads_WK[h](x)  # (batch, seq, d_k)
    V_h = heads_WV[h](x)  # (batch, seq, d_k)
    
    scores = Q_h @ K_h.transpose(-2, -1) / math.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    head_out = weights @ V_h  # (batch, seq, d_k)
    head_outputs.append(head_out)

# 모든 헤드 연결
output_method1 = torch.cat(head_outputs, dim=-1)  # (batch, seq, d_model)
print(f"각 헤드 출력: {head_outputs[0].shape} × {num_heads}개")
print(f"연결 후: {output_method1.shape}")
print("→ 이 방법은 이해하기 쉽지만, 루프가 느립니다")

In [ ]:
# ============================================================
# ⚡ Step 2: 효율적인 Multi-Head 계산 방법

In [ ]:
#
# 핵심 아이디어:
# 하나의 큰 선형 레이어로 모든 헤드의 Q, K, V를 한번에 계산
# 그 다음 reshape으로 헤드 차원을 분리!
#
# 예: d_model=512, num_heads=8, d_k=64
#
# 1. Linear(512, 8*64=512) 로 한번에 계산
#    → shape: (batch, seq, 512)
#
# 2. reshape: (batch, seq, 8, 64)
#    → 8개 헤드, 각각 64차원
#
# 3. permute: (batch, 8, seq, 64)
#    → 헤드 차원을 앞으로 이동 (배치처럼 병렬 처리)
#
# 이렇게 하면 torch.matmul이 (batch*8)개의 attention을 병렬 계산!

print("=" * 60)
print("방법 2: 효율적인 구현 (batch matmul로 병렬 처리)")
print("=" * 60)

d_model = 512
num_heads = 8
d_k = d_model // num_heads  # 64
batch_size = 2
seq_len = 5

# 하나의 큰 W_Q, W_K, W_V
W_Q = nn.Linear(d_model, d_model, bias=False)  # 512 → 512 (8헤드 × 64)
W_K = nn.Linear(d_model, d_model, bias=False)
W_V = nn.Linear(d_model, d_model, bias=False)

x = torch.randn(batch_size, seq_len, d_model)

print("\n[1단계] 한번에 Q, K, V 계산")
Q_all = W_Q(x)  # (batch=2, seq=5, d_model=512)
K_all = W_K(x)
V_all = W_V(x)
print(f"  Q_all shape: {Q_all.shape}  ← (batch, seq, d_model)")

print("\n[2단계] Reshape으로 헤드 분리")
# (batch, seq, d_model) → (batch, seq, num_heads, d_k)
Q_heads = Q_all.view(batch_size, seq_len, num_heads, d_k)
K_heads = K_all.view(batch_size, seq_len, num_heads, d_k)
V_heads = V_all.view(batch_size, seq_len, num_heads, d_k)
print(f"  Q_heads shape: {Q_heads.shape}  ← (batch, seq, num_heads, d_k)")

print("\n[3단계] Permute로 헤드 차원 이동")
# (batch, seq, num_heads, d_k) → (batch, num_heads, seq, d_k)
Q_h = Q_heads.permute(0, 2, 1, 3)
K_h = K_heads.permute(0, 2, 1, 3)
V_h = V_heads.permute(0, 2, 1, 3)
print(f"  Q_h shape: {Q_h.shape}  ← (batch, num_heads, seq, d_k)")
print(f"  → 이제 (batch×num_heads)개를 한번에 병렬 계산!")

print("\n[4단계] 모든 헤드의 Attention 병렬 계산")
scores = torch.matmul(Q_h, K_h.transpose(-2, -1)) / math.sqrt(d_k)
weights = F.softmax(scores, dim=-1)
context = torch.matmul(weights, V_h)
print(f"  context shape: {context.shape}  ← (batch, num_heads, seq, d_k)")

print("\n[5단계] 헤드 결합 (Concatenate)")
# (batch, num_heads, seq, d_k) → (batch, seq, num_heads, d_k)
context = context.permute(0, 2, 1, 3).contiguous()
# (batch, seq, num_heads, d_k) → (batch, seq, d_model)
output_method2 = context.view(batch_size, seq_len, d_model)
print(f"  최종 output shape: {output_method2.shape}  ← (batch, seq, d_model)")

print("\n✅ 두 방법 모두 동일한 shape 출력, 방법2가 훨씬 빠름!")

In [ ]:
# ============================================================
# 🏗️ Step 3: Multi-Head Attention 클래스 완전 구현

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention - 논문 "Attention Is All You Need" 완전 구현
    
    핵심 수식:
    MultiHead(Q,K,V) = Concat(head_1,...,head_h) @ W_O
    where head_i = Attention(Q @ W_Q_i, K @ W_K_i, V @ W_V_i)
    
    구현 핵심:
    - 모든 헤드를 하나의 큰 행렬로 한번에 계산 (효율적!)
    - reshape + permute로 헤드 차원 분리
    - 출력 W_O 레이어로 차원 복원
    """
    
    def __init__(self, d_model, num_heads, dropout=0.0):
        """
        Args:
            d_model:   모델 차원 (임베딩 차원). 논문에서 512
            num_heads: Attention 헤드 수.     논문에서 8
            dropout:   Attention weight에 적용할 드롭아웃 비율
        
        조건: d_model은 num_heads로 나누어 떨어져야 함
        이유: 각 헤드의 차원 = d_model // num_heads (나머지 없어야 함)
        """
        super(MultiHeadAttention, self).__init__()
        
        # 전제 조건 확인
        assert d_model % num_heads == 0, (
            f"d_model({d_model})은 num_heads({num_heads})로 나누어 떨어져야 합니다!"
        )
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 각 헤드의 차원

In [ ]:
# 학습 가능한 가중치 행렬들

In [ ]:
# Q, K, V 투영 레이어
        # 하나의 큰 선형 레이어로 모든 헤드를 한번에 처리
        # nn.Linear(d_model, d_model) → 내부적으로 num_heads개의 헤드
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # d_model → num_heads * d_k
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        
        # 출력 투영 레이어: 모든 헤드 연결 후 → d_model
        # W_O: (num_heads * d_k = d_model) → d_model
        self.W_O = nn.Linear(d_model, d_model)
        
        # Attention weight에 적용할 드롭아웃
        self.dropout = nn.Dropout(dropout)
        
        # 가중치 초기화 (Xavier Uniform: 안정적인 초기값)
        self._init_weights()
    
    def _init_weights(self):
        """Xavier Uniform 초기화로 안정적인 학습 시작"""
        for layer in [self.W_Q, self.W_K, self.W_V]:
            nn.init.xavier_uniform_(layer.weight)
        nn.init.xavier_uniform_(self.W_O.weight)
        if self.W_O.bias is not None:
            nn.init.zeros_(self.W_O.bias)
    
    def split_heads(self, x):
        """
        헤드 분리 헬퍼 함수
        
        (batch, seq_len, d_model) → (batch, num_heads, seq_len, d_k)
        
        과정:
        1. view:    (batch, seq, d_model) → (batch, seq, num_heads, d_k)
        2. permute: (batch, seq, num_heads, d_k) → (batch, num_heads, seq, d_k)
        """
        batch_size, seq_len, _ = x.shape
        
        # d_model을 num_heads × d_k로 분리
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        
        # 헤드 차원을 앞으로 이동 (배치처럼 병렬 처리하기 위해)
        x = x.permute(0, 2, 1, 3)
        
        return x
    
    def combine_heads(self, x):
        """
        헤드 결합 헬퍼 함수
        
        (batch, num_heads, seq_len, d_k) → (batch, seq_len, d_model)
        split_heads의 역과정
        """
        batch_size, _, seq_len, _ = x.shape
        
        # 원래 순서로 복원
        x = x.permute(0, 2, 1, 3).contiguous()
        # contiguous(): permute 후 메모리가 불연속 → 연속으로 만들기
        
        # 헤드 차원 합치기
        x = x.view(batch_size, seq_len, self.d_model)
        
        return x
    
    def forward(self, query, key, value, mask=None):
        """
        Multi-Head Attention 순전파
        
        Args:
            query: (batch, seq_q, d_model) - Query 입력
            key:   (batch, seq_k, d_model) - Key 입력
            value: (batch, seq_v, d_model) - Value 입력 (seq_v = seq_k)
            mask:  (batch, 1, seq_q, seq_k) 또는 None
        
        Self-Attention: query = key = value = x
        Cross-Attention: query = decoder output, key = value = encoder output
        
        Returns:
            output:  (batch, seq_q, d_model)
            weights: (batch, num_heads, seq_q, seq_k) - 각 헤드의 attention weight
        """
        batch_size = query.size(0)

In [ ]:
# 1. Q, K, V 계산 및 헤드 분리

In [ ]:
# 선형 변환 후 헤드 분리
        # shape: (batch, seq, d_model) → (batch, num_heads, seq, d_k)
        Q = self.split_heads(self.W_Q(query))
        K = self.split_heads(self.W_K(key))
        V = self.split_heads(self.W_V(value))

In [ ]:
# 2. Scaled Dot-Product Attention (모든 헤드 동시)

In [ ]:
# Attention Score
        # shape: (batch, num_heads, seq_q, seq_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # 마스크 적용
        if mask is not None:
            scores = scores.masked_fill(mask == 1, float('-inf'))
        
        # Softmax
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        
        # Value 가중 평균
        # shape: (batch, num_heads, seq_q, d_k)
        context = torch.matmul(weights, V)

In [ ]:
# 3. 헤드 결합 후 출력 변환

In [ ]:
# 헤드 결합: (batch, num_heads, seq, d_k) → (batch, seq, d_model)
        context = self.combine_heads(context)
        
        # 출력 선형 변환 W_O
        # shape: (batch, seq, d_model) → (batch, seq, d_model)
        output = self.W_O(context)
        
        return output, weights

In [ ]:
# 테스트

In [ ]:
d_model = 512
num_heads = 8
batch_size = 3
seq_len = 10

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=0.1)

x = torch.randn(batch_size, seq_len, d_model)

mha.eval()
with torch.no_grad():
    output, weights = mha(x, x, x)  # Self-Attention: Q=K=V=x

print("=" * 60)
print("MultiHeadAttention 테스트")
print("=" * 60)
print(f"입력 x:     {x.shape}")
print(f"출력:       {output.shape}")
print(f"가중치:     {weights.shape}")
print()

# 파라미터 수
total = sum(p.numel() for p in mha.parameters())
print(f"총 파라미터 수: {total:,}")
print(f"  W_Q: {d_model}×{d_model} = {d_model*d_model:,}")
print(f"  W_K: {d_model}×{d_model} = {d_model*d_model:,}")
print(f"  W_V: {d_model}×{d_model} = {d_model*d_model:,}")
print(f"  W_O: {d_model}×{d_model}+{d_model}(bias) = {d_model*d_model+d_model:,}")

In [ ]:
# ============================================================
# 📊 Step 4: 각 헤드의 Attention 패턴 시각화

In [ ]:
#
# 각 헤드가 다른 패턴을 학습하는지 확인
# (학습 전이지만 초기 패턴 확인)

# 더 작은 예시로
d_model_small = 32
num_heads_small = 4
seq_len_small = 6
words = ['The', 'cat', 'sat', 'on', 'the', 'mat']

mha_small = MultiHeadAttention(d_model=d_model_small, num_heads=num_heads_small)
torch.manual_seed(0)
x_small = torch.randn(1, seq_len_small, d_model_small)

mha_small.eval()
with torch.no_grad():
    _, weights_small = mha_small(x_small, x_small, x_small)

# 각 헤드의 attention 시각화
fig, axes = plt.subplots(1, num_heads_small, figsize=(5*num_heads_small, 4))

for h in range(num_heads_small):
    head_weights = weights_small[0, h].detach().numpy()
    sns.heatmap(
        head_weights,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        xticklabels=words,
        yticklabels=words,
        ax=axes[h],
        vmin=0, vmax=1,
        cbar=False
    )
    axes[h].set_title(f'Head {h+1}\n(각 헤드 = 다른 패턴)', fontsize=10)
    if h == 0:
        axes[h].set_ylabel('Query')
    axes[h].set_xlabel('Key')

plt.suptitle('Multi-Head Attention: 각 헤드가 다른 관계 패턴을 학습\n(초기화 직후 - 학습 전)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('multi_head_attention.png', dpi=100, bbox_inches='tight')
plt.show()

# 각 헤드의 엔트로피 비교
print("\n각 헤드의 Attention 엔트로피 (높을수록 골고루 분산):")
for h in range(num_heads_small):
    hw = weights_small[0, h].detach().numpy()
    entropy = -np.sum(hw * np.log(hw + 1e-10), axis=-1).mean()
    print(f"  Head {h+1}: {entropy:.3f}")

In [ ]:
# ============================================================
# 🏗️ Step 5: Transformer 인코더 블록 구현

In [ ]:
#
# Multi-Head Attention 위에 나머지 구성요소를 추가하면
# 완전한 Transformer 인코더 블록이 됩니다!
#
# 인코더 블록 구조:
# ┌─────────────────────────────────────────┐
# │  입력 X                                  │
# │     ↓                                    │
# │  Multi-Head Self-Attention               │
# │     ↓                                    │
# │  Add & LayerNorm  ← 잔차 연결            │
# │     ↓                                    │
# │  Feed-Forward Network (FFN)             │
# │     ↓                                    │
# │  Add & LayerNorm  ← 잔차 연결            │
# │     ↓                                    │
# │  출력                                    │
# └─────────────────────────────────────────┘

class TransformerEncoderBlock(nn.Module):
    """
    Transformer 인코더 블록
    
    구성요소:
    1. Multi-Head Self-Attention
    2. Add & Layer Normalization (잔차 연결)
    3. Feed-Forward Network (FFN)
    4. Add & Layer Normalization (잔차 연결)
    
    잔차 연결(Residual Connection):
    output = LayerNorm(x + sublayer(x))
    → x를 더함으로써 gradient vanishing 방지
    → 학습이 더 쉽고 빠름
    
    FFN (Feed-Forward Network):
    Linear(d_model → d_ff) → ReLU → Linear(d_ff → d_model)
    → 각 위치에 독립적으로 비선형 변환 적용
    → d_ff = 4 × d_model (논문 기본값)
    """
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Args:
            d_model:   모델 차원 (논문: 512)
            num_heads: Attention 헤드 수 (논문: 8)
            d_ff:      FFN 중간 차원 (논문: 2048 = 4 × 512)
            dropout:   드롭아웃 비율 (논문: 0.1)
        """
        super(TransformerEncoderBlock, self).__init__()
        
        # ─── 구성요소 1: Multi-Head Self-Attention ───
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # ─── 구성요소 2: Layer Normalization (2개) ───
        # LayerNorm: 각 위치의 특성값을 평균 0, 표준편차 1로 정규화
        # BatchNorm과 달리 시퀀스에서 잘 동작
        self.norm1 = nn.LayerNorm(d_model)  # Attention 후
        self.norm2 = nn.LayerNorm(d_model)  # FFN 후
        
        # ─── 구성요소 3: Feed-Forward Network (FFN) ───
        # 2층 MLP: d_model → d_ff → d_model
        # 각 토큰 위치에 동일하게 독립적으로 적용
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),   # 확장: 512 → 2048
            nn.ReLU(),                   # 비선형 활성화
            nn.Dropout(dropout),         # 드롭아웃
            nn.Linear(d_ff, d_model),   # 축소: 2048 → 512
        )
        
        # ─── 드롭아웃 ───
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        """
        Args:
            x:    입력  shape = (batch, seq_len, d_model)
            mask: 마스크 (패딩 마스크 또는 None)
        
        Returns:
            x:       출력  shape = (batch, seq_len, d_model)
            weights: Attention 가중치
        """
        # ─── Sub-layer 1: Multi-Head Self-Attention ───
        # Self-Attention: Q=K=V=x
        attn_output, weights = self.attention(x, x, x, mask)
        
        # Add & Norm: x + dropout(attn_output) → LayerNorm
        # 잔차 연결: 원래 입력 x를 더해서 gradient 흐름 보장
        x = self.norm1(x + self.dropout(attn_output))
        
        # ─── Sub-layer 2: Feed-Forward Network ───
        ffn_output = self.ffn(x)
        
        # Add & Norm
        x = self.norm2(x + self.dropout(ffn_output))
        
        return x, weights


# 테스트
d_model = 512
num_heads = 8
d_ff = 2048  # 논문에서 4 × d_model
batch_size = 3
seq_len = 10

encoder_block = TransformerEncoderBlock(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    dropout=0.1
)

x = torch.randn(batch_size, seq_len, d_model)

encoder_block.eval()
with torch.no_grad():
    out, wts = encoder_block(x)

print("=" * 60)
print("Transformer 인코더 블록 테스트")
print("=" * 60)
print(f"입력:  {x.shape}")
print(f"출력:  {out.shape}  (shape 변화 없음!)")
print(f"가중치: {wts.shape}")
print()

# 파라미터 계산
total_params = sum(p.numel() for p in encoder_block.parameters())
print(f"인코더 블록 파라미터 수: {total_params:,}")
attn_params = sum(p.numel() for p in encoder_block.attention.parameters())
ffn_params = sum(p.numel() for p in encoder_block.ffn.parameters())
norm_params = sum(p.numel() for p in encoder_block.norm1.parameters()) + \
              sum(p.numel() for p in encoder_block.norm2.parameters())
print(f"  Multi-Head Attention: {attn_params:,}")
print(f"  FFN:                  {ffn_params:,}")
print(f"  LayerNorm (×2):       {norm_params:,}")

In [ ]:
# ============================================================
# 🏗️ Step 6: 완전한 Transformer 인코더 (N개 블록 쌓기)

In [ ]:
class TransformerEncoder(nn.Module):
    """
    완전한 Transformer 인코더
    
    구조:
    입력 → 임베딩 + 위치 인코딩 → N × 인코더블록 → 출력
    
    논문 기본 설정: N=6
    """
    
    def __init__(self, vocab_size, d_model, num_heads, d_ff, 
                 num_layers, max_seq_len, dropout=0.1):
        super(TransformerEncoder, self).__init__()
        
        # 토큰 임베딩
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # 위치 인코딩 (학습 가능한 임베딩 버전 - 간단화)
        # 실제 논문은 sin/cos 위치 인코딩 사용 (067 노트북에서 다룸)
        self.pos_encoding = nn.Embedding(max_seq_len, d_model)
        
        # N개의 인코더 블록 쌓기
        # nn.ModuleList: 여러 모듈을 리스트로 관리
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)  # 마지막 LayerNorm
        
        # 임베딩 가중치 초기화
        nn.init.normal_(self.embedding.weight, mean=0, std=d_model**-0.5)
    
    def forward(self, src_ids, src_mask=None):
        """
        Args:
            src_ids:  토큰 ID  shape = (batch, seq_len)
            src_mask: 패딩 마스크  shape = (batch, 1, 1, seq_len) or None
        
        Returns:
            output: shape = (batch, seq_len, d_model)
        """
        seq_len = src_ids.size(1)
        
        # 위치 인덱스
        positions = torch.arange(seq_len, device=src_ids.device)
        
        # 임베딩 + 위치 인코딩
        # √d_model로 스케일링 (논문에서 임베딩에 적용)
        x = self.embedding(src_ids) * math.sqrt(self.embedding.embedding_dim)
        x = x + self.pos_encoding(positions)
        x = self.dropout(x)
        
        # N개 인코더 블록 순서대로 통과
        all_attentions = []
        for layer in self.layers:
            x, attn = layer(x, src_mask)
            all_attentions.append(attn)
        
        # 최종 LayerNorm
        x = self.norm(x)
        
        return x, all_attentions


# 테스트
vocab_size = 1000
d_model = 256
num_heads = 4
d_ff = 512
num_layers = 3  # 논문은 6
max_seq_len = 128
batch_size = 2
seq_len = 15

encoder = TransformerEncoder(
    vocab_size=vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers,
    max_seq_len=max_seq_len
)

# 임의의 토큰 ID 입력
src_ids = torch.randint(0, vocab_size, (batch_size, seq_len))

encoder.eval()
with torch.no_grad():
    output, all_attn = encoder(src_ids)

print("=" * 60)
print("완전한 Transformer 인코더 테스트")
print("=" * 60)
print(f"입력 토큰 ID: {src_ids.shape}")
print(f"출력 표현:    {output.shape}")
print(f"각 레이어 Attention: {all_attn[0].shape} × {num_layers}개")
print()

total_params = sum(p.numel() for p in encoder.parameters())
print(f"총 파라미터 수: {total_params:,}")
print()
print("모델 구조 요약:")
print(f"  어휘 크기: {vocab_size:,}")
print(f"  모델 차원: {d_model}")
print(f"  Attention 헤드: {num_heads}")
print(f"  FFN 차원: {d_ff}")
print(f"  레이어 수: {num_layers}")

## 📋 정리

### ✅ 이번 실습에서 배운 것

1. **Multi-Head Attention의 필요성**
   - 단일 헤드는 한 가지 관계만 학습
   - 여러 헤드 → 다양한 관계를 동시에 포착

2. **효율적인 구현**
   - 하나의 큰 선형 레이어 + reshape/permute
   - GPU 병렬 연산 최대 활용

3. **Transformer 인코더 블록**
   - Multi-Head Attention + FFN + LayerNorm + Residual
   - N개 블록을 쌓아 완전한 인코더 구성

4. **핵심 하이퍼파라미터**
   - d_model=512, num_heads=8, d_ff=2048, N=6 (원 논문)

### ➡️ 다음 실습 (067, 068)
- **BERT**: Masked Language Modeling으로 사전학습
- 인코더 구조를 이용한 양방향 언어 이해 모델